In [1]:
import pandas as pd
import gensim
import nltk
from gensim import corpora
from nltk.corpus import stopwords

In [24]:
df = pd.read_csv('disaster_train.csv')
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        7613 non-null   int64 
 1   keyword   7552 non-null   object
 2   location  5080 non-null   object
 3   text      7613 non-null   object
 4   target    7613 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 297.5+ KB


In [27]:
df = df.fillna('')
df.isnull().sum()

id          0
keyword     0
location    0
text        0
target      0
dtype: int64

In [28]:
df['text'].head()

0    Our Deeds are the Reason of this #earthquake M...
1               Forest fire near La Ronge Sask. Canada
2    All residents asked to 'shelter in place' are ...
3    13,000 people receive #wildfires evacuation or...
4    Just got sent this photo from Ruby #Alaska as ...
Name: text, dtype: object

In [29]:
df['text'] = df['text'].str.lower();
df['text'] = df['text'].str.replace('[^a-z ]',' ', regex=True)
df['text']

0       our deeds are the reason of this  earthquake m...
1                  forest fire near la ronge sask  canada
2       all residents asked to  shelter in place  are ...
3              people receive  wildfires evacuation or...
4       just got sent this photo from ruby  alaska as ...
                              ...                        
7608    two giant cranes holding a bridge collapse int...
7609     aria ahrary  thetawniest the out of control w...
7610    m            utc   km s of volcano hawaii  htt...
7611    police investigating after an e bike collided ...
7612    the latest  more homes razed by northern calif...
Name: text, Length: 7613, dtype: object

In [31]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [32]:
stop_words = set(stopwords.words('English'))

In [33]:
def clean_words(sentence):
    words = sentence.split()
    result =[]
    for w in words:
        if w not in stop_words and len(w)>2:
            result.append(w)
    return result;

df['tokens'] = df['text'].apply(clean_words)

In [34]:
df['tokens']

0        [deeds, reason, earthquake, may, allah, forgive]
1               [forest, fire, near, ronge, sask, canada]
2       [residents, asked, shelter, place, notified, o...
3       [people, receive, wildfires, evacuation, order...
4       [got, sent, photo, ruby, alaska, smoke, wildfi...
                              ...                        
7608    [two, giant, cranes, holding, bridge, collapse...
7609    [aria, ahrary, thetawniest, control, wild, fir...
7610            [utc, volcano, hawaii, http, zdtoyd, ebj]
7611    [police, investigating, bike, collided, car, l...
7612    [latest, homes, razed, northern, california, w...
Name: tokens, Length: 7613, dtype: object

In [48]:
dictionary = corpora.Dictionary(df['tokens'])
print(dictionary)

Dictionary<21345 unique tokens: ['allah', 'deeds', 'earthquake', 'forgive', 'may']...>


In [49]:
corpus = []
for tokens in df['tokens']:
    bow = dictionary.doc2bow(tokens);
    corpus.append(bow)

In [50]:
lda = gensim.models.LdaModel(corpus,id2word=dictionary,num_topics=3,passes=10);

In [51]:
for idx,data in lda.print_topics():
    print(f" Topic : {idx} : {data}")

 Topic : 0 : 0.054*"http" + 0.007*"suicide" + 0.004*"news" + 0.004*"new" + 0.004*"bomber" + 0.003*"crash" + 0.003*"bombing" + 0.003*"get" + 0.002*"body" + 0.002*"reddit"
 Topic : 1 : 0.028*"http" + 0.008*"like" + 0.007*"https" + 0.005*"one" + 0.004*"amp" + 0.003*"video" + 0.003*"fire" + 0.003*"time" + 0.003*"get" + 0.003*"still"
 Topic : 2 : 0.055*"http" + 0.005*"disaster" + 0.005*"amp" + 0.004*"nuclear" + 0.003*"via" + 0.003*"wildfire" + 0.003*"storm" + 0.003*"news" + 0.003*"california" + 0.003*"obama"
